# Module 01 — Repository Types

A snapshot repository is the **storage backend** where Elasticsearch writes snapshot data.
Choosing the right repository type is the first decision you make in any snapshot strategy.

This module covers every repository type available in Elasticsearch 9.x:

| Type | Backend | Use case |
|------|---------|----------|
| `fs` | Local / shared filesystem | Development, on-prem NFS |
| `url` | Read-only HTTP/S | Distributing snapshots to consumers |
| `source` | Wraps another repo | Minimal-size source-only backups |
| `s3` | AWS S3 (or compatible) | Cloud / MinIO |
| `gcs` | Google Cloud Storage | GCP deployments |
| `azure` | Azure Blob Storage | Azure deployments |

In [1]:
import os, subprocess, time
from helpers import *

es = get_client()
wait_for_green(es)

✓ Cluster health: yellow

---
## 1. Filesystem Repository (`fs`)

The simplest repository type. Elasticsearch writes snapshot files to a directory that must be:
- Listed in `path.repo` in `elasticsearch.yml` (already configured in this training env)
- Accessible from every node (NFS / shared storage in multi-node clusters)

### Key settings

| Setting | Default | Description |
|---------|---------|-------------|
| `location` | *(required)* | Path on disk |
| `compress` | `true` | Compress metadata files |
| `chunk_size` | unlimited | Split large files |
| `max_snapshot_bytes_per_sec` | `40mb` | Throttle creation |
| `max_restore_bytes_per_sec` | unlimited | Throttle restore |
| `readonly` | `false` | Prevent writes |

In [2]:
heading('1. Filesystem repository — register')

# Already registered in Module 00, but we recreate with explicit settings here
es.snapshot.create_repository(
    name='fs-demo',
    body={
        'type': 'fs',
        'settings': {
            'location': '/usr/share/elasticsearch/snapshots',
            'compress': True,
            'max_snapshot_bytes_per_sec': '40mb',
            'max_restore_bytes_per_sec': '100mb',
        },
    },
)
success('Repository fs-demo registered')

─────────────────────────────────────── 1. Filesystem repository — register ───────────────────────────────────────

✓ Repository fs-demo registered

In [3]:
# Verify connectivity
v = es.snapshot.verify_repository(name='fs-demo')
pp(dict(v), 'POST /_snapshot/fs-demo/_verify')

╭─ POST /_snapshot/fs-demo/_verify ─╮
│ {                                 │
│   "nodes": {                      │
│     "S_FtdbGzTsOR3WqKn8wt3g": {   │
│       "name": "es01"              │
│     }                             │
│   }                               │
│ }                                 │
╰───────────────────────────────────╯

In [4]:
# Take a quick snapshot so we can inspect the files on disk
delete_snapshot_if_exists(es, 'fs-demo', 'fs-demo-snap-01')
es.snapshot.create(
    repository='fs-demo',
    snapshot='fs-demo-snap-01',
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': False},
    wait_for_completion=True,
)
success('Snapshot created')

ℹ Deleted existing snapshot 'fs-demo-snap-01'

✓ Snapshot created

In [5]:
# Inspect the raw files written to disk
# (The snapshot-repo dir is bind-mounted to ./snapshot-repo/ on your host)
heading('Raw snapshot files on disk')
result = subprocess.run(['find', '/usr/share/elasticsearch/snapshots', '-type', 'f'],
                        capture_output=True, text=True)
for line in sorted(result.stdout.strip().split('\n')):
    console.print(f'  {line}')

info('''
File types:
  index-N        — repository metadata (list of all snapshots)
  snap-<uuid>.dat — snapshot metadata (which indices, shards, settings)
  meta-<uuid>.dat — global cluster state metadata
  indices/<uuid>/ — per-index data, one subdir per shard
''')

─────────────────────────────────────────── Raw snapshot files on disk ────────────────────────────────────────────

ℹ 
File types:
  index-N        — repository metadata (list of all snapshots)
  snap-<uuid>.dat — snapshot metadata (which indices, shards, settings)
  meta-<uuid>.dat — global cluster state metadata
  indices/<uuid>/ — per-index data, one subdir per shard

---
## 2. Read-Only URL Repository (`url`)

A read-only view of a snapshot repository served over HTTP/S (or `file://`, `ftp://`).
Useful for **distributing** snapshots to consumer clusters without giving them write access.

We simulate this by serving the `fs-demo` repository directory with Python's built-in HTTP server.

In [6]:
import subprocess, time

# The ES container can reach Jupyter on the Docker network via its container name.
# We'll use a file:// URL pointing directly at the mounted path — the simplest approach.
heading('2. Read-only URL repository')

# Note: file:// URLs require the path to already be in path.repo (it is)
es.snapshot.create_repository(
    name='url-demo',
    body={
        'type': 'url',
        'settings': {
            'url': 'file:///usr/share/elasticsearch/snapshots',
        },
    },
)
success('URL repository registered')

─────────────────────────────────────────── 2. Read-only URL repository ───────────────────────────────────────────

✓ URL repository registered

In [7]:
# List snapshots visible through the URL repo (same data as fs-demo)
snaps = es.snapshot.get(repository='url-demo', snapshot='*')
for s in snaps['snapshots']:
    console.print(f"  {s['snapshot']}  state={s['state']}  indices={s['indices']}")

info('URL repository is read-only — creating a snapshot against it will fail.')

source-snap-01  state=SUCCESS  indices=['kibana_sample_data_ecommerce']

baseline  state=SUCCESS  indices=['kibana_sample_data_flights', '.apm-custom-link', 
'kibana_sample_data_ecommerce', '.kibana_task_manager_9.3.0_001', '.kibana_alerting_cases_9.3.0_001', 
'.kibana_security_solution_9.3.0_001', '.kibana_ingest_9.3.0_001', '.kibana_analytics_9.3.0_001', 
'.kibana_locks-000001', '.kibana_9.3.0_001', '.kibana_search_solution_9.3.0_001', '.apm-agent-configuration', 
'.kibana_security_session_1', '.kibana_usage_counters_9.3.0_001', '.ds-kibana_sample_data_logs-2026.04.06-000001']

fs-demo-snap-01  state=SUCCESS  indices=['kibana_sample_data_ecommerce']

ℹ URL repository is read-only — creating a snapshot against it will fail.

In [8]:
# Demonstrate read-only enforcement
try:
    es.snapshot.create(
        repository='url-demo',
        snapshot='should-fail',
        body={'indices': []},
        wait_for_completion=True,
    )
except Exception as exc:
    warn(f'Expected error: {exc}')
    success('Confirmed: URL repository is read-only.')

⚠ Expected error: ApiError(500, 'repository_exception', ' cannot create snapshot in a readonly repository')

✓ Confirmed: URL repository is read-only.

---
## 3. Source-Only Repository (`source`)

A source-only repository **wraps another repository** and stores only `_source` fields —
no inverted indices, no doc values, no norms. This can reduce snapshot size by **up to 50%**.

### Trade-offs

| | Regular snapshot | Source-only snapshot |
|--|--|--|
| Storage | Full | ~50% smaller |
| Restore speed | Fast | Slow (re-indexes from source) |
| Restored index is... | Fully queryable | Read-only; only `match_all` + scroll |
| Requires `_source` enabled | No | **Yes** |
| Works with synthetic source | Yes | **No** |

In [ ]:
heading('3. Source-only repository')

es.snapshot.create_repository(
    name='source-demo',
    body={
        'type': 'source',
        'settings': {
            'delegate_type': 'fs',
            'location': '/usr/share/elasticsearch/snapshots',
        },
    },
)
success('Source-only repository registered')

In [ ]:
delete_snapshot_if_exists(es, 'source-demo', 'source-snap-01')
es.snapshot.create(
    repository='source-demo',
    snapshot='source-snap-01',
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': False},
    wait_for_completion=True,
)
success('Source-only snapshot created')

In [ ]:
# Restore into a new index name and observe the read-only restriction
try:
    es.indices.delete(index='ecommerce-source-restored', ignore_unavailable=True)
except Exception:
    pass

es.snapshot.restore(
    repository='source-demo',
    snapshot='source-snap-01',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'rename_pattern': 'kibana_sample_data_(.*)',
        'rename_replacement': '$1-source-restored',
        'include_global_state': False,
    },
    wait_for_completion=True,
)

# match_all works
result = es.search(index='ecommerce-source-restored', body={'query': {'match_all': {}}, 'size': 1})
success(f'match_all hit count: {result["hits"]["total"]["value"]}')

# But indexing fails
try:
    es.index(index='ecommerce-source-restored', body={'test': 'write'})
except Exception as exc:
    warn(f'Expected: cannot write to source-restored index: {type(exc).__name__}')

In [ ]:
# Cleanup
es.indices.delete(index='ecommerce-source-restored', ignore_unavailable=True)

---
## 4. S3-Compatible Repository (`s3`) — MinIO

The `s3` repository type works with any S3-compatible object store.
We use **MinIO** (running as a Docker service alongside ES) so no AWS account is needed.

### Key settings

| Setting | Description |
|---------|-------------|
| `bucket` | S3 bucket name |
| `endpoint` | Custom endpoint URL (for MinIO / non-AWS) |
| `path_style_access` | `true` for MinIO / path-style endpoints |
| `protocol` | `http` or `https` |
| `base_path` | Prefix within the bucket |
| `client` | Named client config from the keystore |
| `server_side_encryption` | AES256 at rest |
| `storage_class` | `standard`, `intelligent_tiering`, etc. |

In [ ]:
import boto3
from botocore.client import Config

heading('4. S3 repository — create MinIO bucket')

s3 = boto3.client(
    's3',
    endpoint_url=f'http://{MINIO_ENDPOINT}',
    aws_access_key_id=MINIO_ROOT_USER,
    aws_secret_access_key=MINIO_ROOT_PASSWORD,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

bucket_name = 'es-snapshots'
existing = [b['Name'] for b in s3.list_buckets().get('Buckets', [])]
if bucket_name not in existing:
    s3.create_bucket(Bucket=bucket_name)
    success(f'Bucket "{bucket_name}" created in MinIO')
else:
    info(f'Bucket "{bucket_name}" already exists')

In [ ]:
# Credentials are stored in the ES keystore as s3.client.default.access_key / secret_key
# (added via: echo "value" | docker exec -i es01 bin/elasticsearch-keystore add --stdin <key>)
# In production, always use the keystore — never put credentials in repository settings.
es.snapshot.create_repository(
    name='s3-minio-demo',
    body={
        'type': 's3',
        'settings': {
            'bucket': bucket_name,
            'endpoint': MINIO_INTERNAL_ENDPOINT,
            'path_style_access': True,
            'protocol': 'http',
            'compress': True,
            'base_path': 'training',
        },
    },
)
success('S3/MinIO repository registered')

In [ ]:
v = es.snapshot.verify_repository(name='s3-minio-demo')
pp(dict(v), 'Verify s3-minio-demo')

In [ ]:
delete_snapshot_if_exists(es, 's3-minio-demo', 's3-snap-01')
es.snapshot.create(
    repository='s3-minio-demo',
    snapshot='s3-snap-01',
    body={'indices': ['kibana_sample_data_flights'], 'include_global_state': False},
    wait_for_completion=True,
)
success('S3 snapshot created')

# List objects stored in MinIO
objects = s3.list_objects_v2(Bucket=bucket_name, Prefix='training/')
info(f'Objects in MinIO bucket: {objects["KeyCount"]}')
for obj in objects.get('Contents', [])[:10]:
    console.print(f"  {obj['Key']}  ({obj['Size']} bytes)")

---
## 5. Cloud Repository Types — GCS & Azure (Conceptual)

GCS and Azure repositories work identically in structure to S3 but use cloud-provider-specific
authentication and endpoints. No live cloud account is needed for this section.

### GCS (`gcs`)

```python
es.snapshot.create_repository(
    name='my-gcs-repo',
    body={
        'type': 'gcs',
        'settings': {
            'bucket': 'my-gcs-bucket',
            'client': 'default',           # reads credentials_file from keystore
            'base_path': 'es-snapshots',
            'compress': True,
            'project_id': 'my-gcp-project',
        },
    },
)
```

Credentials go in the keystore as:
```
elasticsearch-keystore add-file gcs.client.default.credentials_file /path/to/service-account.json
```

### Azure (`azure`)

```python
es.snapshot.create_repository(
    name='my-azure-repo',
    body={
        'type': 'azure',
        'settings': {
            'client': 'default',            # reads account + key from keystore
            'container': 'es-snapshots',
            'base_path': 'training',
            'compress': True,
            'location_mode': 'primary_only',
        },
    },
)
```

Credentials:
```
elasticsearch-keystore add azure.client.default.account
elasticsearch-keystore add azure.client.default.key
```

---
## 6. Repository Verification, Analysis & Cleanup

Elasticsearch provides three management APIs for repositories:

In [ ]:
heading('6a. Repository verification')
# Checks that every node can access the repository
v = es.snapshot.verify_repository(name=FS_REPO_NAME)
pp(dict(v), f'POST /_snapshot/{FS_REPO_NAME}/_verify')

In [ ]:
heading('6b. Repository analysis')
# Performs a series of read/write/delete operations to assess repository performance
# blob_count and max_blob_size control the test intensity
analysis = es.snapshot.analyze_repository(
    name=FS_REPO_NAME,
    blob_count=10,
    max_blob_size='1mb',
    timeout='60s',
)
pp(dict(analysis), f'POST /_snapshot/{FS_REPO_NAME}/_analyze')

In [ ]:
heading('6c. Repository cleanup')
# Removes stale data left by partially-completed or deleted snapshots
cleanup = es.snapshot.cleanup_repository(name=FS_REPO_NAME)
pp(dict(cleanup), f'POST /_snapshot/{FS_REPO_NAME}/_cleanup')

In [ ]:
heading('All registered repositories')
all_repos = es.snapshot.get_repository(name='*')
t = Table('Name', 'Type', 'Location / Bucket')
for name, repo in all_repos.items():
    loc = repo['settings'].get('location') or repo['settings'].get('bucket', '—')
    t.add_row(name, repo['type'], loc)
console.print(t)

## Summary

| Repository | When to use |
|-----------|-------------|
| `fs` | Development, on-prem with shared storage |
| `url` | Read-only distribution of snapshots |
| `source` | Minimal-footprint backups; archival |
| `s3` | AWS or any S3-compatible store (MinIO, Ceph) |
| `gcs` | Google Cloud deployments |
| `azure` | Azure deployments |

**Next:** [02_what_is_in_a_snapshot.ipynb](02_what_is_in_a_snapshot.ipynb)